# MovieSense — DistilBERT training on Google Colab

Trains the report's proposed model (DistilBERT + mean pooling + linear multi-label head) on the **same
processed dataset, labels, splits and short-text windows** as the local TF-IDF baseline, then:

1. tunes per-genre thresholds on validation (same procedure as the baseline),
2. embeds all 42,306 plots with the fine-tuned encoder for retrieval and calibrates `tau_match`,
3. runs the Integrated Gradients deletion test,
4. runs the final test-split comparison against the baseline (H1, paired bootstrap),
5. zips everything for `scripts/install_distilbert_artifacts.sh`.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Expected time on a T4: roughly
30–45 min for one seed with 3 epochs (estimate, not measured). Nothing in this notebook reports a result
unless the corresponding cell actually ran.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Upload the bundle
On your Mac run `./scripts/package_colab_bundle.sh`, then upload `dist/moviesense_colab_bundle.zip` here
(or put it in Google Drive and change `BUNDLE` to its Drive path after mounting).

In [ ]:
import os, zipfile
BUNDLE = "/content/moviesense_colab_bundle.zip"
if not os.path.exists(BUNDLE):
    from google.colab import files
    up = files.upload()            # choose dist/moviesense_colab_bundle.zip
    BUNDLE = "/content/" + next(iter(up))
os.makedirs("/content/MovieSense", exist_ok=True)
zipfile.ZipFile(BUNDLE).extractall("/content/MovieSense")
%cd /content/MovieSense
!ls

## 3. Install dependencies (torch is preinstalled on Colab)

In [ ]:
!pip install -q -r requirements-ml.txt

## 4. Record the shipped dataset and splits, then build the metadata store
The processed dataset and split ID files come from the local pipeline. The SQLite store (titles, cast)
is rebuilt here from the official corpus; if the rebuilt dataset differs in any way (e.g. a different
library version), the shipped dataset and splits are restored so training uses exactly the local splits.

In [ ]:
import hashlib, json, shutil, pathlib
def sha(p): return hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()
shipped = {p: sha(p) for p in ["data/processed/dataset.jsonl.gz", "data/processed/labels.json",
           "data/processed/splits/train.txt", "data/processed/splits/val.txt", "data/processed/splits/test.txt"]}
shutil.copytree("data/processed", "/content/shipped_processed", dirs_exist_ok=True)
shutil.copy("reports/data_report.json", "/content/shipped_data_report.json")
for p, h in shipped.items(): print(h[:16], p)

In [ ]:
!./scripts/download_data.sh
!python scripts/build_dataset.py
rebuilt = {p: sha(p) for p in shipped}
same = rebuilt == shipped
print("rebuilt dataset identical to shipped:", same)
if not same:   # keep the rebuilt store, restore the shipped dataset + splits + report
    for p in shipped: shutil.copy("/content/shipped_processed/" + p.split("data/processed/")[1], p)
    shutil.copy("/content/shipped_data_report.json", "reports/data_report.json")
assert {p: sha(p) for p in shipped} == shipped, "splits must match the local pipeline"
print("OK: training will use the local split definitions")

## 5. Train DistilBERT
One seed by default. For the report's three-seed protocol use `--seeds 42 13 7` (≈3× the time; free Colab
sessions may disconnect). Model selection and thresholds use validation only.

In [ ]:
!python scripts/train_distilbert.py --seeds 42 --epochs 3 --batch-size 32 --lr 3e-5 --max-length 256 2>&1 | tee train_distilbert.log

## 6. Embed all plots with the fine-tuned encoder and evaluate retrieval

In [ ]:
!python scripts/build_index.py --encoder distilbert 2>&1 | tee build_index_distilbert.log

## 7. Integrated Gradients deletion test

In [ ]:
!python scripts/eval_explanations.py --model distilbert --n 300 2>&1 | tee eval_explanations_distilbert.log

## 8. Final test-split comparison with the baseline (H1)

In [ ]:
!python scripts/evaluate.py 2>&1 | tee evaluate.log

## 9. Sanity check: load the saved bundle and predict one sentence

In [ ]:
import sys; sys.path.insert(0, "src")
from moviesense.models.transformer import TransformerModel
m = TransformerModel.load("models/distilbert")
s = "A young detective investigates a series of mysterious murders while uncovering a conspiracy involving his closest friend."
p = m.predict_proba([s])[0]
print(sorted(zip(m.labels, p.round(3)), key=lambda t: -t[1])[:5])
print(m.explain(s, [m.labels[int(p.argmax())]])[m.labels[int(p.argmax())]]["top_words"])

## 10. Download the artefacts

In [ ]:
!zip -qr moviesense_distilbert_artifacts.zip models/distilbert reports/distilbert_validation.json \
    reports/retrieval_distilbert.json reports/explain_distilbert.json reports/final_comparison.json \
    train_distilbert.log build_index_distilbert.log eval_explanations_distilbert.log evaluate.log
!ls -lh moviesense_distilbert_artifacts.zip
from google.colab import files
files.download("moviesense_distilbert_artifacts.zip")

## Integrate locally
```bash
./scripts/install_distilbert_artifacts.sh ~/Downloads/moviesense_distilbert_artifacts.zip
.venv/bin/pip install --no-cache-dir -r requirements-ml.txt
MOVIESENSE_MODEL=distilbert MOVIESENSE_RETRIEVAL=distilbert ./scripts/run_backend.sh
```
The install script refuses artefacts trained on a different dataset hash.